# King Domino Project - Overview

<!-- **Process so far:**
- Data Preprocessing: Filter out duplicate images, analyze data for patterns such as exact same tile-pattern.
- Ground Truth's (GT):
    - GT_PointScore
- Splitting Strategy
- Quantify terrains into 20-vector(Water, Field, Forest, etc.) using Color HSV Histograms and SVM/kNN multi-class classification.


**Remaining project steps:**
- Ground Truth's (GT):
    GT_BoardTileMatrix
    GT_CrownPerTile
    GT_CrownPerBoard
- Identifying crown multipliers using Template Matching.
- Building a 2D-array logic to calculate board scores and verify against manually calculated ground truth.
- Assessing performance utilizing Confusion Matrices and final scoring metrics. -->

### Architecture Overview
The project is structured as a computer vision pipeline designed to digitize and score a physical King Domino board. The pipeline consists of five main stages:
1. **Grid Splitting:** Cropping full board images into a 5x5 logical grid of individual tiles.
2. **Feature Extraction:** Processing raw tiles to extract color histograms (HSV) and save them for model building/analysis.
3. **Tile Classification:** Identifying the terrain type (e.g., Forest, Water, Field) of each tile using color masks or texture fallbacks.
4. **Crown Detection:** Counting the number of scoring multipliers (crowns) present on each classified terrain tile.
5. **Score Calculation:** Evaluating the 5x5 logical grid using a Breadth-First Search (BFS) algorithm to calculate final connected-region scores according to game rules.

### Current Status

| Module | Status | Description |
| :--- | :---: | :--- |
| `board_split.py` | 🟢 **Done** | Correctly divides the main image into a 5x5 tile grid. |
| `feature_extraction.py` | 🟢 **Done** | Extracts HSV histograms from tiles and builds a dataset (`features.csv`). |
| `extract_tile_feature_vector.py` | 🟡 **WIP** | Successfully loads images, splits tiles, and orchestrates feature extraction, but lacks the final inference and scoring pipeline. |
| `classify_tile.py` | 🔴 **TODO** | Stub (`TileClassifier`). Needs logic to convert HSV coverage into a terrain class string. |
| `detect_crown.py` | 🔴 **TODO** | Stub (`CrownDetector`). Needs contour/shape detection to count 0-3 golden crowns per tile. |
| `scoring.py` | 🔴 **TODO** | Stub (`ScoreCalculator`). Needs the BFS algorithm to multiply connected region sizes by crown counts. |

### Missing Parts
1. **Classification Logic:** Translating the extracted HSV features/masks into definitive terrain labels (`classify_tile.py`).
2. **Crown Counting Logic:** Identifying specific yellow contours/shapes on non-empty tiles to act as multipliers (`detect_crown.py`).
3. **Game Rule Engine:** Implementing the BFS grid traversal to group adjacent matching terrains and calculate the final score (`scoring.py`).
4. **End-to-End Integration:** Modifying `extract_tile_feature_vector.py` to use the classifiers and pass the resulting 5x5 grid of objects to the scoring engine.

### Recommended Next Step
**Implement Terrain Classification (`classify_tile.py`)**
Before we can count crowns or calculate the final score, the system needs to know *what* each tile is. Since the feature extraction for HSV histograms is already complete (`features.csv`), we have the data needed to build the classification logic.

**Next Code to Generate:**
We should populate the `classify()` method inside `TileClassifier` (`classify_tile.py`) to map cell images to terrain string labels using color bounding/mask coverage.


## Project Components

1. **Data Preprocessing & Splitting Strategy** 

2. **Tile Classification:** 
- Iterate over the 5x5 extracted tiles.
- Extract features into a 20-value vector (HSV histogram logic).
- Perform multi-class terrain classification (SVM / kNN).

3. **Crown Detection:** 
- Operate on individual tiles to locate crown icons.
- Utilize template matching (e.g., using `high_res_crown.png` / `low_res_crown.png`).
- Processe grayscale/blobs, handling transparent "Don't care" pixels.

4. **Game Logic (Ground Truth):** 
- Build a 2D-array representing the 5x5 grid.
- Evaluate connected components and calculates final score based on Kingdomino rules:
  *(e.g., 6 connected water tiles * 2 crowns = 12 points).*

5. **Evaluation:**

## 1. Data Preprocessing & Splitting Strategy
To prevent data leakage, the split is performed strictly on a board-level rather than a tile-level. Paired images (with and without 3D objects) stay in the same split.

* **Total Boards:** 36 unique game boards.
* **Invalid Data:** Images 71, 73, and 74 are excluded, because they are duplicates of images 68, 69, 70
* **Hold-out Test Set:** 6 board groups (12 images in total: e.g., (1,5), (19,23), etc.). Set aside completely for the final evaluation to test robustness.
* **Training & Validation (Grouped CV):** The remaining 30 board groups are used for 5-fold grouped cross-validation.
  * Each fold contains exactly 6 specific board groups.
  * Extracted tiles always inherit the split assignment of their parent board.
  * 3D properties act as a robustness condition, not a split axis.
* **Data Extraction:** ~1775 tiles (71 valid images × 25 fields).